In [ ]:
import sqlite3
import random
import pandas as pd

# ============================================================
# SQL and Pandas Data Analysis: Employee Database
# Name: Obilisetti Ravi Kiran
# Student ID: iitp_aimlt_26011062
# ============================================================


# ============================================================
# TASK 1: Create Database
# ============================================================
# Set seed for reproducibility
random.seed(42)
# Define data parameters
departments = ["Engineering", "Sales", "Marketing", "HR", "Finance"]
num_employees = 40
# Connect to (or create) the database
conn = sqlite3.connect("employees.db")
cursor = conn.cursor()
# Drop table if it already exists to avoid duplicates on re-run
cursor.execute("DROP TABLE IF EXISTS employees")
# Create the employees table
cursor.execute("""
    CREATE TABLE employees (
        emp_id            INTEGER PRIMARY KEY,
        name              TEXT,
        department        TEXT,
        salary            REAL,
        years_experience  INTEGER,
        performance_score REAL
    )
""")
# Generate and insert 40 synthetic employee records
records = []
for i in range(1, num_employees + 1):
    emp_id            = i
    name              = f"Employee_{i}"
    department        = random.choice(departments)
    salary            = round(random.uniform(50_000, 150_000), 2)
    years_experience  = random.randint(1, 15)
    performance_score = round(random.uniform(1.0, 5.0), 2)
    records.append((emp_id, name, department, salary,
                    years_experience, performance_score))
cursor.executemany("""
    INSERT INTO employees
        (emp_id, name, department, salary, years_experience, performance_score)
    VALUES (?, ?, ?, ?, ?, ?)
""", records)
conn.commit()
print(f"Database created successfully with {len(records)} employee records.\n")
# Quick preview of the full table
df_all = pd.read_sql_query("SELECT * FROM employees", conn)
print("Full Employee Table:")
print(df_all.to_string(index=False))

Database created successfully with 40 employee records.

Full Employee Table:
 emp_id        name  department    salary  years_experience  performance_score
      1  Employee_1 Engineering  52501.08                 5               1.98
      2  Employee_2       Sales 123647.12                11               3.96
      3  Employee_3     Finance  58693.88                 7               1.13
      4  Employee_4 Engineering  71863.80                 9               3.41
      5  Employee_5     Finance  69883.77                11               3.81
      6  Employee_6          HR  72044.06                10               2.11
      7  Employee_7 Engineering 125880.74                 3               3.79
      8  Employee_8   Marketing  77787.13                 4               4.83
      9  Employee_9   Marketing  60221.03                 7               1.39
     10 Employee_10   Marketing 110372.60                13               1.17
     11 Employee_11          HR 103622.81            

In [3]:
# ============================================================
# TASK 2: SQL Queries
# ============================================================
# Query 1: High-performing & experienced employees
query1 = """
    SELECT name, department, salary, performance_score
    FROM   employees
    WHERE  performance_score >= 4.0
      AND  years_experience  >= 3
    ORDER BY performance_score DESC
    LIMIT 15
"""
df_query1 = pd.read_sql_query(query1, conn)
print("\n--- Query 1: High Performers with 3+ Years Experience (Top 15) ---")
print(df_query1.to_string(index=False))

# Query 2: Mid-salary employees in Engineering or Sales
query2 = """
    SELECT *
    FROM   employees
    WHERE  salary BETWEEN 70000 AND 110000
      AND  department IN ('Engineering', 'Sales')
    ORDER BY department ASC, salary DESC
"""
df_query2 = pd.read_sql_query(query2, conn)
print("\n--- Query 2: Mid-Salary Employees in Engineering or Sales ---")
print(df_query2.to_string(index=False))

# Query 3: Headcount and average salary per department
query3 = """
    SELECT department,
           COUNT(*)       AS employee_count,
           ROUND(AVG(salary), 2) AS avg_salary
    FROM   employees
    GROUP BY department
    ORDER BY avg_salary DESC
"""
df_query3 = pd.read_sql_query(query3, conn)
print("\n--- Query 3: Employee Count & Average Salary by Department ---")
print(df_query3.to_string(index=False))


--- Query 1: High Performers with 3+ Years Experience (Top 15) ---
       name  department    salary  performance_score
Employee_27     Finance  89940.05               4.99
Employee_19       Sales  96226.02               4.96
 Employee_8   Marketing  77787.13               4.83
Employee_38       Sales 103937.90               4.69
Employee_22          HR  76774.09               4.65
Employee_31          HR 102911.43               4.44
Employee_20     Finance  71961.52               4.37
Employee_36 Engineering 137051.86               4.37
Employee_40   Marketing 137872.19               4.23
Employee_29 Engineering  65284.13               4.17

--- Query 2: Mid-Salary Employees in Engineering or Sales ---
 emp_id        name  department    salary  years_experience  performance_score
     39 Employee_39 Engineering 109894.46                 8               1.08
     15 Employee_15 Engineering  88012.62                 8               3.54
     34 Employee_34 Engineering  79350.01        

In [4]:
# ============================================================
# TASK 3: Pandas Analysis
# ============================================================
# 3a: Average performance_score by department using groupby()
avg_perf_by_dept = (
    df_all
    .groupby("department")["performance_score"]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"performance_score": "avg_performance_score"})
    .sort_values("avg_performance_score", ascending=False)
)
print("\n--- Pandas 3a: Average Performance Score by Department ---")
print(avg_perf_by_dept.to_string(index=False))

# 3b: Replicate Query 1 using ONLY Pandas
df_query1_pandas = (
    df_all
    .loc[
        (df_all["performance_score"] >= 4.0) &   # AND condition with &
        (df_all["years_experience"]  >= 3),
        ["name", "department", "salary", "performance_score"]  # column selection
    ]
    .sort_values("performance_score", ascending=False)  # ORDER BY performance_score DESC
    .head(15)                                            # LIMIT 15
    .reset_index(drop=True)
)
print("\n--- Pandas 3b: Replication of Query 1 Using Only Pandas ---")
print(df_query1_pandas.to_string(index=False))

# 3c: SQL vs Pandas Comparison (see Markdown cell below)
comparison_text = """
## SQL vs Pandas Comparison

**Syntax Differences**
SQL uses a declarative, English-like syntax: equality is `=`, logical
combination is `AND`/`OR`, and sorting is `ORDER BY col DESC`.  
Pandas uses Python expressions: equality is `==`, logical combination
inside `.loc[]` uses `&`/`|` with parentheses around each condition,
and sorting is `.sort_values("col", ascending=False)`.

**When to Use Each Tool**
SQL is ideal when data already lives in a relational database, when
multiple tables need joining, or when the query will be shared across
teams or applications.  Pandas shines for exploratory data analysis,
feature engineering, visualisation pipelines, and when the workflow is
already in Python/Jupyter.

**Large Datasets**
For datasets too large to fit in memory, SQL is the better choice.  A
database engine stores data on disk and processes only the rows and
columns needed, streaming results to the client.  Pandas loads the
entire dataset into RAM, which becomes impractical at scale.  Tools
such as Dask or PySpark extend the Pandas API to distributed data, but
for standard out-of-memory workloads a well-indexed SQL database
remains the most practical and efficient solution.
"""
print(comparison_text)

# Close the database connection
conn.close()
print("Database connection closed.")


--- Pandas 3a: Average Performance Score by Department ---
 department  avg_performance_score
      Sales                   4.34
Engineering                   3.31
         HR                   3.30
    Finance                   3.23
  Marketing                   2.50

--- Pandas 3b: Replication of Query 1 Using Only Pandas ---
       name  department    salary  performance_score
Employee_27     Finance  89940.05               4.99
Employee_19       Sales  96226.02               4.96
 Employee_8   Marketing  77787.13               4.83
Employee_38       Sales 103937.90               4.69
Employee_22          HR  76774.09               4.65
Employee_31          HR 102911.43               4.44
Employee_20     Finance  71961.52               4.37
Employee_36 Engineering 137051.86               4.37
Employee_40   Marketing 137872.19               4.23
Employee_29 Engineering  65284.13               4.17

## SQL vs Pandas Comparison

**Syntax Differences**
SQL uses a declarative, English-l

# Task 3c: SQL vs Pandas Comparison

**Syntax Differences**  
SQL uses a declarative, English-like syntax: equality is `=`, logical combination is `AND`/`OR`, and sorting is `ORDER BY col DESC`.  
Pandas uses Python expressions: equality is `==`, logical combination inside `.loc[]` uses `&`/`|` with parentheses around each condition, and sorting is `.sort_values("col", ascending=False)`.

**When to Use Each Tool**  
SQL is ideal when data already lives in a relational database, when multiple tables need joining, or when the query will be shared across teams or applications.  
Pandas shines for exploratory data analysis, feature engineering, visualisation pipelines, and when the workflow is already in Python/Jupyter.

**Large Datasets**  
For datasets too large to fit in memory, SQL is the better choice.  A database engine stores data on disk and processes only the rows and columns needed, streaming results to the client.  Pandas loads the entire dataset into RAM, which becomes impractical at scale.  Tools such as Dask or PySpark extend the Pandas API to distributed data, but for standard out-of-memory workloads a well-indexed SQL database remains the most practical and efficient solution.